
# Swiggy Instamart – Customer Intelligence & Support Copilot


## Project Overview

This project demonstrates an end-to-end customer intelligence and
AI-powered support application using synthetic grocery retail data.

The project is divided into four parts:

1. Data Foundation
2. Customer Segmentation and Lapse Prediction
3. Support Ticket Triage with Guardrails
4. Policy RAG Assistant and Refund Agent

## Technologies

- Python
- Pandas and NumPy
- SQL and SQLite
- Scikit-learn
- TF-IDF and Cosine Similarity
- Jupyter Notebook
- Generative AI simulation

## Project Setup

This notebook imports reusable functions from the `src` directory.

All data processing, model development, classification,
retrieval and agent logic are maintained in separate Python modules.

In [49]:
from pathlib import Path
import sys
import sqlite3
import json

import pandas as pd
import numpy as np

from IPython.display import display

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
RAW = DATA / "raw"
CLEAN = DATA / "clean"
OUTPUTS = ROOT / "outputs"
STATE = ROOT / "state"

db_path = DATA / "instamart.db"

print("Project root:", ROOT)
print("Database exists:", db_path.exists())
print("Outputs folder exists:", OUTPUTS.exists())

Project root: c:\instamart_customer_copilot
Database exists: True
Outputs folder exists: True


## 1. Part A – Data Foundation

## Objective
Prepare reliable, structured data for customer and business analysis.

- Data quality inspection
- Missing value and duplicate detection
- Data cleaning and standardization
- Data validation
- SQLite database creation
- SQL analysis
- Exploratory Data Analysis

### Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [65]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


In [66]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Data Cleaning

The raw customer, product and order datasets are cleaned using
the reusable functions in `data_pipeline.py`.

The cleaned datasets are saved under `data/clean/`.

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.


In [62]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

In [ ]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


ALL CHECKPOINTS PASSED


### SQL Analysis

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

SQL queries are used to analyze:

- Customer spending
- Revenue by city and product category
- Product sales performance
- Customer purchase frequency
- Customer lifetime spending
- City-wise top customers
- Daily cumulative revenue

The queries use the SQLite database created during data preparation.

In [59]:
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        con
    )

display(tables)

,name
0,customers
1,orders
2,products


In [60]:
with sqlite3.connect(db_path) as con:
    for table in ["customers", "products", "orders"]:
        print(f"\n{table.upper()}")
        df = pd.read_sql_query(
            f"SELECT * FROM {table} LIMIT 5",
            con
        )
        display(df)


CUSTOMERS


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female
3,C0004,Bengaluru,2024-06-21,24,female
4,C0005,Bengaluru,2024-10-01,25,female



PRODUCTS


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245
3,P004,Sunflower Oil 1L,Grocery,155
4,P005,Sugar 1kg,Grocery,48



ORDERS


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
1,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
2,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23
3,ORD-100004,C0001,P023,2,2025-07-29 22:27:32,completed,20,336.00,33
4,ORD-100005,C0002,P022,2,2025-07-18 15:52:20,completed,0,90.00,17


### Exploratory Data Analysis

EDA helps understand customer purchasing behaviour, revenue
distribution, order patterns and relationships between variables.

Key observations include:

- City-wise revenue performance
- Average and median order value
- Order amount distribution and outliers
- Correlation between numerical variables
- Daily revenue trends

## 2. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [67]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary

,k,inertia,silhouette
0,2,1820.761,0.465
1,3,1268.235,0.445
2,4,743.071,0.526
3,5,617.367,0.480
4,6,503.171,0.482
5,7,408.460,0.416
6,8,366.503,0.412


,segment_id,size,recency,frequency,monetary
0,0,436,28.86,3.77,1057.62
1,1,138,24.12,6.77,7129.70
2,2,419,8.77,19.40,3410.02
3,3,182,109.04,2.20,685.99


PCA explained variance ratio: [0.63427875 0.21803762] total: 0.8523163667502166


,customer_id,frequency,monetary,recency,segment_id,segment_name,is_premium
0,C0001,4,512.85,14,0,Regular Customers,False
1,C0002,7,4644.90,28,1,Premium Customers,True
2,C0003,22,3644.85,7,2,Loyal Active Customers,False
3,C0004,5,369.00,0,0,Regular Customers,False
4,C0005,2,324.00,147,3,Occasional Customers,False


{'cohort_rows': 1156,
 'lapsed': 443,
 'baseline_accuracy': 0.6163793103448276,
 'models': {'logistic': {'best_params': {'model__C': 100},
   'cv_f1': np.float64(0.650552962900789),
   'accuracy': 0.728448275862069,
   'precision': 0.6444444444444445,
   'recall': 0.651685393258427,
   'f1': 0.6480446927374302,
   'confusion_matrix': [[111, 32], [31, 58]]},
  'forest': {'best_params': {'model__max_depth': 8,
    'model__min_samples_leaf': 10},
   'cv_f1': np.float64(0.6865517479789224),
   'accuracy': 0.7198275862068966,
   'precision': 0.6333333333333333,
   'recall': 0.6404494382022472,
   'f1': 0.6368715083798883,
   'confusion_matrix': [[110, 33], [32, 57]]}},
 'retention_list':     customer_id  lapse_probability  actually_lapsed
 58        C0525           0.603686                1
 190       C0267           0.585974                0
 57        C0082           0.580447                0
 227       C0060           0.570635                0
 16        C0167           0.550253         

In [69]:
from pathlib import Path

ROOT = Path.cwd().parent
OUTPUTS = ROOT / "outputs"

expected_files = [
    "customer_segments.csv",
    "lapse_model.joblib",
    "retention_list.csv"
]

for filename in expected_files:
    file_path = OUTPUTS / filename
    print(f"{filename}: {'Created' if file_path.exists() else 'Missing'}")

customer_segments.csv: Created
lapse_model.joblib: Created
retention_list.csv: Created


Inertia generally decreases as the number of clusters increases because more clusters allow customers to be grouped more closely around their cluster centres. However, choosing K=8 simply because it has lower inertia may create too many small, less meaningful segments. Therefore, we should also consider the elbow method, silhouette score, and business interpretability to select the most suitable K.

### Lapse Prediction

--Features: Use orders before 1 September 2025 only.
--Target label: Check whether the customer placed any completed order from 1–30 September 2025.
--Cohort: Customers with at least one completed order before 1 September 2025.

In [3]:
rfm, diagnostics, profile, pca_variance = rfm_and_segments(k=4)

display(diagnostics)
display(profile)

print("PCA explained variance:", pca_variance)

,k,inertia,silhouette
0,2,1820.760553,0.464972
1,3,1268.234587,0.445161
2,4,743.071108,0.525544
3,5,617.366636,0.480495
4,6,503.170622,0.481872
5,7,408.460454,0.416337
6,8,366.503487,0.412329


,segment_id,size,recency,frequency,monetary
0,0,436,28.862385,3.772936,1057.618234
1,1,138,24.115942,6.768116,7129.695652
2,2,419,8.766110,19.403341,3410.020525
3,3,182,109.043956,2.203297,685.992582


PCA explained variance: [0.63427875 0.21803762]


## 3.Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

### Input Guardrails and Privacy Protection

Empty tickets are skipped.

Tickets exceeding the maximum input token limit are flagged.

Email addresses, phone numbers and order IDs are redacted
before sending ticket text to the model.

In [70]:
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
{'tokens': 12520, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4}}


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Occasional Customers,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Loyal Active Customers,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,Regular Customers,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,,Loyal Active Customers,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Regular Customers,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Loyal Active Customers,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Loyal Active Customers,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Loyal Active Customers,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium Customers,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,Regular Customers,NORMAL,support_queue,78


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,skipped_empty,4
4,flagged_long,4


## Comparison

In [71]:
from src.triage import run_pipeline, ZERO
import pandas as pd

# Run zero-shot classification
zero_results, zero_summary = run_pipeline(
    prompt=ZERO,
    token_budget=100000
)

# Load actual ticket categories
tickets = pd.read_csv(
    ROOT / "data" / "raw" / "tickets.csv",
    keep_default_na=False
)

actual = tickets.set_index("ticket_id")["true_category"]

def evaluate_accuracy(results):
    classified = results[
        results["status"].isin([
            "classified_json",
            "classified_keyword"
        ])
    ].copy()

    classified["actual_category"] = (
        classified["ticket_id"].map(actual)
    )

    accuracy = (
        classified["category"] == classified["actual_category"]
    ).mean() * 100

    return len(classified), accuracy

# Evaluate both runs
zero_count, zero_accuracy = evaluate_accuracy(zero_results)
few_count, few_accuracy = evaluate_accuracy(triage_results)

comparison = pd.DataFrame({
    "Prompt": ["Zero-shot", "Few-shot"],
    "Classified Tickets": [zero_count, few_count],
    "Accuracy (%)": [zero_accuracy, few_accuracy],
    "Total Tokens": [
        zero_summary["tokens"],
        triage_summary["tokens"]
    ]
})

display(comparison.round(2))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s


,Prompt,Classified Tickets,Accuracy (%),Total Tokens
0,Zero-shot,147,85.71,7372
1,Few-shot,147,100.00,12520


### Generate zero-shot confusion matrix

In [25]:
import pandas as pd

# Keep successfully classified zero-shot tickets
zero_classified = zero_results[
    zero_results["status"].isin([
        "classified_json",
        "classified_keyword"
    ])
].copy()

# Add actual categories
zero_classified["actual_category"] = (
    zero_classified["ticket_id"].map(actual)
)

# Create confusion matrix
confusion_matrix = pd.crosstab(
    zero_classified["actual_category"],
    zero_classified["category"],
    rownames=["Actual"],
    colnames=["Predicted"],
    dropna=False
)

display(confusion_matrix)

Predicted,billing,delivery,fraud,technical
Actual,,,,
billing,38,0,0,0
delivery,0,36,0,0
fraud,11,0,23,0
technical,10,0,0,29


### Calculate tokens per ticket

In [72]:
print("Zero-shot tokens per classified ticket:",
      round(zero_summary["tokens"] / zero_count, 2))

print("Few-shot tokens per classified ticket:",
      round(triage_summary["tokens"] / few_count, 2))

Zero-shot tokens per classified ticket: 50.15
Few-shot tokens per classified ticket: 85.17


### Compare zero-shot and few-shot accuracy

In [ ]:
budget_results, budget_summary = run_pipeline(
    prompt=ZERO,
    token_budget=6000
)

print("Budget summary:")
print(budget_summary)

print("\nBudget status counts:")
display(budget_results["status"].value_counts())

,Prompt,Classified Tickets,Accuracy (%),Total Tokens
0,Zero-shot,147,85.71,7372
1,Few-shot,147,100.00,12520


### Budget demonstration

In [27]:
budget_results, budget_summary = run_pipeline(
    prompt=ZERO,
    token_budget=6000
)

print("Budget demonstration:")
print(budget_summary)

display(budget_results["status"].value_counts())

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
Budget demonstration:
{'tokens': 6009, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 111, 'budget_exhausted': 29, 'classified_keyword': 9, 'flagged_invalid_output': 7, 'skipped_empty': 4, 'flagged_long': 4}}


status
classified_json           111
budget_exhausted           29
classified_keyword          9
flagged_invalid_output      7
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

### Key Findings

- 164 support tickets were processed.
- Four empty tickets were skipped.
- Four long tickets were flagged.
- 135 responses were validated as JSON.
- 12 responses were validated using keyword fallback.
- Nine invalid outputs were sent for human review.
- Few-shot classification achieved 100% accuracy on the
  147 accepted classifications.
- Token-budget controls successfully flagged tickets
  for human review.

The final pipeline supports privacy-aware and cost-conscious
customer support ticket routing.

In [76]:
final_triage = pd.read_csv(OUTPUTS / "triage_results.csv")
budget_demo = pd.read_csv(OUTPUTS / "triage_budget_demo.csv")

print("Final triage rows:", len(final_triage))
print("Budget demo rows:", len(budget_demo))

display(final_triage.head())

Final triage rows: 164
Budget demo rows: 164


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Occasional Customers,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Loyal Active Customers,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,Regular Customers,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,NaN,Loyal Active Customers,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Regular Customers,HIGH,security_team,84


## 4.(i) Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.


The system combines:

- Policy document chunking
- TF-IDF retrieval
- Cosine similarity
- Grounded answers
- Policy evaluation
- Database-backed tools
- Refund calculations
- Human escalation
- Agent execution limits
- Checkpoint persistence

In [74]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


## 4.(ii) Part D — refund tools and safe stopping
### Timeout Safety

The agent has a time limit.

If the execution time exceeds the configured limit,
the agent stops and hands the request to a human agent.



### Refund Ledger and Checkpoint Verification

Refund and escalation actions are recorded in a JSONL ledger.

Agent execution checkpoints are saved as JSON files.

These records provide a simple audit trail of the simulated
refund workflow.

In [75]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent
db_path = ROOT / "data" / "instamart.db"

print("Database path:", db_path)
print("Database exists:", db_path.exists())
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

Database path: c:\instamart_customer_copilot\data\instamart.db
Database exists: True
Tools: ['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']
Example order lookup:
ORD-100048 {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65}
Refund calculation: {'refund_inr': 250.0}
Timeout safety demo: {'status': 'FALLBACK_TIMEOUT', 'final_answer': 'FALLBACK_TIMEOUT: hand off to a human agent.', 'steps': [], 'llm_calls': 0, 'tokens_used': 0}


In [45]:
agent_result = run_agent(
    f"Refund order {example_order}",
    scenario_id="normal-refund-demo"
)

print("Status:", agent_result["status"])
print("Final answer:", agent_result["final_answer"])
print("LLM calls:", agent_result["llm_calls"])
print("Tokens used:", agent_result["tokens_used"])

print("\nTool execution sequence:")
display(pd.DataFrame(agent_result["steps"]))

Status: REFUND
Final answer: REFUND: INR 128.25 issued.
LLM calls: 5
Tokens used: 837

Tool execution sequence:


,tool,arguments,observation
0,get_order,{'order_id': 'ORD-100048'},"{'order_id': 'ORD-100048', 'status': 'complete..."
1,search_policy,{'query': 'refund policy late delivery'},"{'results': [{'source': 'refund_policy.txt', '..."
2,calculate_refund,"{'order_amount': 128.25, 'delivery_minutes': 65}",{'refund_inr': 128.25}
3,issue_refund,"{'order_id': 'ORD-100048', 'refund_inr': 128.25}",{'refund_inr': 128.25}


### Key Findings

- Policy documents were chunked and indexed using TF-IDF.
- Relevant policy passages were retrieved using cosine similarity.
- Grounded answers were generated from retrieved context.
- Policy retrieval and answer evaluation were performed.
- SQLite-backed order lookup was successful.
- Refund calculation and simulated refund issuance were successful.
- Timeout fallback was demonstrated.
- Refund actions and agent checkpoints were stored locally.

The application demonstrates how policy retrieval and
tool-based AI workflows can support customer service operations.

## Final generated artifacts and reproducibility check


In [46]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['customer_segments.csv', 'lapse_model.joblib', 'refund_ledger.jsonl', 'retention_list.csv', 'triage_budget_demo.csv', 'triage_results.csv']
\nstate
['5ce09dea-4293-4352-8be2-888b267a17b9.json']
Database exists: True


# Final Project Summary

The Instamart Customer Intelligence & Support Copilot
combines data analytics, machine learning and AI techniques
into one integrated application.

## Completed Project Areas

### Part A – Data Foundation
Data cleaning, validation, SQLite and SQL analysis.

### Part B – Customer Intelligence
RFM segmentation, customer profiling and lapse prediction.

### Part C – Support Ticket Triage
PII redaction, prompt comparison, output validation,
token budgets and ticket routing.

### Part D – AI Support Copilot
Policy retrieval, grounded answers, refund calculations,
tool execution and checkpoint persistence.

## Conclusion

The project demonstrates an integrated approach to
understanding customer behaviour and improving customer
support operations using data-driven and AI-assisted methods.